In [ ]:
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Store-Sales-Forecasting"

CLEANED_PATH = f"{PROJECT_PATH}/Data/Cleaned"
PROCESSED_PATH = f"{PROJECT_PATH}/Data/Processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)

Mounted at /content/drive


In [11]:
import pandas as pd

train_cleaned = pd.read_csv(
    f"{CLEANED_PATH}/train_cleaned.csv",
    parse_dates=["date"]
)

stores_cleaned = pd.read_csv(
    f"{CLEANED_PATH}/stores_cleaned.csv"
)

transactions_cleaned = pd.read_csv(
    f"{CLEANED_PATH}/transactions_cleaned.csv",
    parse_dates=["date"]
)

oil_cleaned = pd.read_csv(
    f"{CLEANED_PATH}/oil_cleaned.csv",
    parse_dates=["date"]
)

holidays_cleaned = pd.read_csv(
    f"{CLEANED_PATH}/holidays_cleaned.csv",
    parse_dates=["date"]
)

test_cleaned = pd.read_csv(
    f"{CLEANED_PATH}/test_cleaned.csv",
    parse_dates=["date"]
)

print("All cleaned datasets loaded successfully.")
print("Train shape:", train_cleaned.shape)

All cleaned datasets loaded successfully.
Train shape: (3000888, 6)


In [12]:
master_df = train_cleaned.merge(
    stores_cleaned,
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

print("Rows before merge:", train_cleaned.shape[0])
print("Rows after merge:", master_df.shape[0])

print("\nMissing values in new store columns:")
print(master_df[["city", "state", "type", "cluster"]].isnull().sum())

Rows before merge: 3000888
Rows after merge: 3000888

Missing values in new store columns:
city       0
state      0
type       0
cluster    0
dtype: int64


In [13]:
# Merge daily store transactions
master_df = master_df.merge(
    transactions_cleaned,
    on=["date", "store_nbr"],
    how="left",
    validate="many_to_one"
)

print("Rows after transaction merge:", master_df.shape[0])

print("\nMissing values in transactions:")
print(master_df["transactions"].isnull().sum())

print("\nPreview:")
master_df.head()

Rows after transaction merge: 3000888

Missing values in transactions:
245784

Preview:


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,NaN
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,NaN
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,NaN
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,NaN
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,NaN


In [14]:
print(master_df.shape)
print(master_df["transactions"].isnull().sum())

(3000888, 11)
245784


In [15]:
# Merge oil prices
master_df = master_df.merge(
    oil_cleaned,
    on="date",
    how="left",
    validate="many_to_one"
)

print("Rows after oil merge:", master_df.shape[0])

print("\nMissing values in oil price:")
print(master_df["dcoilwtico"].isnull().sum())

master_df.head()

Rows after oil merge: 3000888

Missing values in oil price:
857142


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,NaN,93.14
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,NaN,93.14
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,NaN,93.14
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,NaN,93.14
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,NaN,93.14


In [16]:
print(master_df.shape)
print(master_df["dcoilwtico"].isnull().sum())

(3000888, 12)
857142


In [17]:
# Create one holiday record per date
holiday_daily = holidays_cleaned.groupby("date").agg(
    holiday_type=("type", lambda x: ", ".join(x.astype(str).unique())),
    holiday_locale=("locale", lambda x: ", ".join(x.astype(str).unique())),
    holiday_description=("description", lambda x: " | ".join(x.astype(str).unique())),
    is_transferred=("transferred", "max")
).reset_index()

# Create a simple holiday indicator
holiday_daily["is_holiday"] = 1

print("Original holiday rows:", holidays_cleaned.shape[0])
print("Unique holiday dates:", holiday_daily.shape[0])

holiday_daily.head()

Original holiday rows: 350
Unique holiday dates: 312


,date,holiday_type,holiday_locale,holiday_description,is_transferred,is_holiday
0,2012-03-02,Holiday,Local,Fundacion de Manta,False,1
1,2012-04-01,Holiday,Regional,Provincializacion de Cotopaxi,False,1
2,2012-04-12,Holiday,Local,Fundacion de Cuenca,False,1
3,2012-04-14,Holiday,Local,Cantonizacion de Libertad,False,1
4,2012-04-21,Holiday,Local,Cantonizacion de Riobamba,False,1


In [18]:
master_df = master_df.merge(
    holiday_daily,
    on="date",
    how="left",
    validate="many_to_one"
)

print("Rows after holiday merge:", master_df.shape[0])
print("\nMissing values in is_holiday before filling:")
print(master_df["is_holiday"].isnull().sum())

Rows after holiday merge: 3000888

Missing values in is_holiday before filling:
2551824


In [19]:
master_df["is_holiday"] = master_df["is_holiday"].fillna(0).astype(int)

print(master_df["is_holiday"].value_counts())
print("\nFinal shape:", master_df.shape)

is_holiday
0    2551824
1     449064
Name: count, dtype: int64

Final shape: (3000888, 17)


In [20]:
# Check missing values after all merges
missing_summary = master_df.isnull().sum().sort_values(ascending=False)

print(missing_summary[missing_summary > 0])

holiday_locale         2551824
is_transferred         2551824
holiday_description    2551824
holiday_type           2551824
dcoilwtico              857142
transactions            245784
dtype: int64


In [21]:
# Flag whether transaction data was missing
master_df["transactions_missing"] = master_df["transactions"].isnull().astype(int)

# Fill missing transactions with 0 for the dashboard/EDA dataset
# We will review this choice again before model training.
master_df["transactions"] = master_df["transactions"].fillna(0)

# Oil price: carry the most recent available price forward, then backward-fill any beginning gaps
master_df = master_df.sort_values("date")
master_df["dcoilwtico"] = master_df["dcoilwtico"].ffill().bfill()

# Holiday text fields: missing means no listed event
master_df["holiday_type"] = master_df["holiday_type"].fillna("No Holiday")
master_df["holiday_locale"] = master_df["holiday_locale"].fillna("No Holiday")
master_df["holiday_description"] = master_df["holiday_description"].fillna("No Holiday")

# Verify
print(master_df.isnull().sum()[master_df.isnull().sum() > 0])
print("Final shape:", master_df.shape)

is_transferred    2551824
dtype: int64
Final shape: (3000888, 18)


In [22]:
master_df["is_transferred"] = master_df["is_transferred"].fillna(False)

/tmp/ipykernel_21301/3117172511.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master_df["is_transferred"] = master_df["is_transferred"].fillna(False)


In [23]:
print(master_df.isnull().sum()[master_df.isnull().sum() > 0])
print("Final shape:", master_df.shape)

Series([], dtype: int64)
Final shape: (3000888, 18)


In [24]:
master_df.to_csv(
    f"{PROCESSED_PATH}/master_dataset.csv",
    index=False
)

print("Master dataset saved successfully.")

Master dataset saved successfully.


In [26]:
master_df.to_csv(
    "/content/drive/MyDrive/Store-Sales-Forecasting/Data/Processed/master_dataset.csv",
    index=False
)

print("Master dataset saved as CSV.")

Master dataset saved as CSV.
